In [31]:
from pathlib import Path

from lassi.profiler import Timer, MultiProfiler, GPUProfiler
from lassi.source_file import SourceFile
from typing import Annotated


from openai import AsyncOpenAI
from groq import Groq


from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

## Minimal C Example

In [ ]:
# Declare a new file

source_file = SourceFile(
    file_name = Path("test.c")
)

source_file

In [ ]:
source_file.write( # this could be the LLM output
    """#include <stdio.h>

        int main(void) {
            printf("Compiler test successful!");
            return 0;
        }
    """
)

executable = source_file.compile()
success = source_file.execute()
report = source_file.get_execution_report()

In [ ]:
report # default report is latency

## More complex CUDA example

In [24]:
source_file = SourceFile(
    file_name = Path("similarity_cuda_test.cu"),
    folder_path = Path("/home/gbrun/test_cuda_folder/src/") # <-- from another folder
)

In [25]:
source_file # this time lang is CUDA

Source File:
   file name: similarity_cuda_test.cu,
   folder path: /home/gbrun/test_cuda_folder/src,
   language:    Language.CUDA,
   executable:  None

In [26]:
source_file.compile(
    kwds="-O3", # all flags go here
    output_file=Path("test_cuda") # custom name
    ) 

Compiling /home/gbrun/test_cuda_folder/src/similarity_cuda_test.cu using nvcc...
Compiling with command: nvcc /home/gbrun/test_cuda_folder/src/similarity_cuda_test.cu -O3 -o /home/gbrun/LASSI-TOOLS/test_cuda


In [ ]:
source_file.execute(
    args="100 100", # need args
    profiler=Timer() # GPU profiler. There is NVIDIA and AMD
    )

Running with command: /home/gbrun/LASSI-TOOLS/test_cuda 100 100


CompletedProcess(args=[PosixPath('/home/gbrun/LASSI-TOOLS/test_cuda'), '100', '100'], returncode=0, stdout='------------------------\nH(X;Y) : 8.00000000\nH(X)   : 8.00000000\nH(Y)   : 8.00000000\nMI(X;Y): 8.00000000\n------------------------\nTime to move to device:  240.730534 ms\nTime to compute histograms: 6.591772 ms\nTime to gather: 0.089120 ms\nTime compute reduce: 0.023000 ms\nTime to compute H(X;Y):  0.226561 ms\n------------------------\nH(X;Y) : 8.00000000\nH(X)   : 8.00000000\nH(Y)   : 8.00000000\nMI(X;Y): 8.00000000\n------------------------\nTime to move to device:  17.242192 ms\nTime to compute histograms: 2.052724 ms\nTime to gather: 0.081000 ms\nTime compute reduce: 0.005240 ms\nTime to compute H(X;Y):  0.218201 ms\n------------------------\nH(X;Y) : 8.00000000\nH(X)   : 8.00000000\nH(Y)   : 8.00000000\nMI(X;Y): 8.00000000\n------------------------\nTime to move to device:  17.107032 ms\nTime to compute histograms: 2.046244 ms\nTime to gather: 0.079960 ms\nTime compute

In [30]:
source_file.get_execution_report() # more complex report

TimerReport(latency=2.1694021322764456)

## Code Generation

In [32]:
import dotenv
from dataclasses import dataclass
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel

from openai import AsyncOpenAI

In [33]:
dotenv.load_dotenv()

True

In [39]:
source_file = SourceFile(
    file_name = Path("llm_generated_code.c"),
)

In [46]:
@dataclass
class CodeGenDependencies:
    language: Annotated[str, Field(description="Language to implement the code in.")]

@dataclass
class CodeGenOutput:
    code: Annotated[str, Field(description="Valid code that follows the requested task. Code only.")]

In [ ]:
MODEL_NAME = "openai/gpt-oss-120b"

model = GroqModel(MODEL_NAME)

# local vLLM example
# model = OpenAIChatModel(MODEL_NAME, provider=OpenAIProvider(base_url="http://localhost:8000/v1"))

agent = Agent(
    model=model,
    instructions=(
        "You are a helpful coding tool."
    ),
    output_type= CodeGenOutput,
)

In [61]:
deps = CodeGenDependencies(language=source_file.lang.value)

result = await agent.run("Write me a C program that takes from args a number N and prints the first N args of the fibonacci sequence.")

source_file.write(result.output.code)

In [62]:
print(source_file.read())

#include <stdio.h>
#include <stdlib.h>

int main(int argc, char *argv[])
{
    if (argc != 2) {
        fprintf(stderr, "Usage: %s N\n", argv[0]);
        return 1;
    }

    char *endptr;
    long n = strtol(argv[1], &endptr, 10);
    if (*endptr != '\0' || n < 0) {
        fprintf(stderr, "Invalid number: %s\n", argv[1]);
        return 1;
    }

    if (n == 0) {
        return 0; // nothing to print
    }

    unsigned long long a = 0, b = 1;
    for (long i = 0; i < n; ++i) {
        if (i == 0) {
            printf("%llu", a);
        } else if (i == 1) {
            printf(" %llu", b);
        } else {
            unsigned long long c = a + b;
            printf(" %llu", c);
            a = b;
            b = c;
        }
    }
    printf("\n");
    return 0;
}



In [63]:
source_file.compile()
output = source_file.execute(args="10")
source_file.get_execution_report()

Compiling /home/gbrun/LASSI-TOOLS/llm_generated_code.c using gcc...
Compiling with command: gcc /home/gbrun/LASSI-TOOLS/llm_generated_code.c -o /home/gbrun/LASSI-TOOLS/llm_generated_code
Running with command: /home/gbrun/LASSI-TOOLS/llm_generated_code 10


TimerReport(latency=0.0018172026611864567)